# 2.07 时序差分学习：TD、Sarsa 与 Q-learning

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.07_td_learning.ipynb)


## 为什么要学

TD（Temporal-Difference）学习是 RL 中最核心、最有特色的思想：**用"对未来的估计"来更新"对现在的估计"**（称为 bootstrapping，自举）。不用等回合结束，每走一步就能学。

- DQN = Q-learning + 神经网络；
- SAC / TD3 / RLT 的 critic 都用 TD 目标训练；
- PPO 的 GAE 由 TD 误差加权组合而成。

对应书中第 7 章。

---

## 1. TD(0)：估计状态值

贝尔曼方程说 $v_\pi(s)=\mathbb{E}[R_{t+1}+\gamma v_\pi(S_{t+1})\mid S_t=s]$。这是一个"求期望"的问题，用上一课的随机近似：

$$
v(s_t)\leftarrow v(s_t)-\alpha\Big[v(s_t)-\underbrace{\big(r_{t+1}+\gamma v(s_{t+1})\big)}_{\text{TD 目标 }\bar v_t}\Big]
$$

$$
\delta_t = v(s_t) - \big(r_{t+1}+\gamma v(s_{t+1})\big)\qquad\text{（TD 误差）}
$$

（书中把 TD 误差定义为 $v - \bar v$；很多论文定义成相反符号 $\delta_t = r+\gamma v(s')-v(s)$。读代码时要注意符号。）

**TD 与 MC 的对比**：

| | MC | TD |
|---|---|---|
| 目标 | 完整回报 $g_t$ | $r_{t+1}+\gamma v(s_{t+1})$ |
| 何时更新 | 回合结束后 | 每一步 |
| 偏差 | 无偏 | 有偏（用了估计值 $v(s_{t+1})$） |
| 方差 | 大（累积了整条轨迹的随机性） | 小（只有一步的随机性） |
| 适用于持续性任务 | 不方便 | 可以 |

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, policy_to_matrix

env = GridWorld()
gamma = 0.9
rng = np.random.default_rng(0)

def true_v(env, pi, gamma):
    return np.linalg.solve(np.eye(env.n_states) - gamma * np.einsum("sa,sat->st", pi, env.P), (pi * env.R).sum(1))

def v_star_q_star(env, gamma):
    v = np.zeros(env.n_states)
    for _ in range(1000):
        q = env.R + gamma * env.P @ v
        v = q.max(1)
    return v, q

V_STAR, Q_STAR = v_star_q_star(env, gamma)

### 练习 1：TD(0) 策略评估

评估均匀随机策略。智能体在一条很长的轨迹上不断走，每一步都更新一次 $v$。

In [ ]:
def td0_evaluate(env, pi, gamma, alpha=0.05, n_steps=200_000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.zeros(env.n_states)
    s = env.reset(int(rng.integers(env.n_states)))
    for t in range(n_steps):
        a = rng.choice(env.n_actions, p=pi[s])
        ns, r = env.step(a)
        # TODO: TD(0) 更新 v[s]
        raise NotImplementedError
        s = ns
    return v

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def td0_evaluate(env, pi, gamma, alpha=0.05, n_steps=200_000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.zeros(env.n_states)
    s = env.reset(int(rng.integers(env.n_states)))
    for t in range(n_steps):
        a = rng.choice(env.n_actions, p=pi[s])
        ns, r = env.step(a)
        td_target = r + gamma * v[ns]
        v[s] = v[s] - alpha * (v[s] - td_target)
        s = ns
    return v

In [ ]:
# ✅ 自测
pi_rand = np.full((25, 5), 0.2)
v_td = td0_evaluate(env, pi_rand, gamma)
v_true = true_v(env, pi_rand, gamma)
err = np.abs(v_td - v_true).mean()
print(f"TD(0) 与真实值的平均误差 {err:.3f}（真实值范围 {v_true.min():.1f} ~ {v_true.max():.1f}）")
assert err < 0.5
print("通过 ✅")

---

## 2. Sarsa：估计动作值并改进策略

为了改进策略，需要动作值。把 TD 的思想用在 $q$ 上：对经验 $(s_t,a_t,r_{t+1},s_{t+1},a_{t+1})$（名字 SARSA 就是这五个字母）：

$$
q(s_t,a_t)\leftarrow q(s_t,a_t)-\alpha\Big[q(s_t,a_t)-\big(r_{t+1}+\gamma q(s_{t+1},a_{t+1})\big)\Big]
$$

它在求解动作值的贝尔曼方程 $q_\pi(s,a)=\mathbb{E}[R+\gamma q_\pi(S',A')\mid s,a]$。

**Sarsa 控制算法**：每一步用 Sarsa 更新 $q$，并立刻把策略更新为关于 $q$ 的 ε-greedy 策略（截断到极致的广义策略迭代）。

**变体**：

- **Expected Sarsa**：目标中用 $\sum_{a'}\pi(a'|s_{t+1})q(s_{t+1},a')$ 代替 $q(s_{t+1},a_{t+1})$，方差更小；
- **n 步 Sarsa**：目标用 $n$ 步真实奖励再加上 $\gamma^n q(s_{t+n},a_{t+n})$。$n=1$ 是 Sarsa，$n=\infty$ 是 MC。这是 TD 与 MC 之间的折中，PPO 的 GAE 是它的加权平均版本。

## 3. Q-learning：直接学最优动作值

Q-learning 的更新：

$$
q(s_t,a_t)\leftarrow q(s_t,a_t)-\alpha\Big[q(s_t,a_t)-\big(r_{t+1}+\gamma\max_{a}q(s_{t+1},a)\big)\Big]
$$

它在求解**贝尔曼最优方程**（动作值形式）$q(s,a)=\mathbb{E}[R+\gamma\max_{a'}q(S',a')\mid s,a]$。

**on-policy vs off-policy**：

- **行为策略**（behavior policy）：用来与环境交互、产生数据的策略；
- **目标策略**（target policy）：我们想要学习/评估的策略。

Sarsa 的目标中用的 $a_{t+1}$ 来自行为策略，两者必须相同，所以是 **on-policy**。Q-learning 的目标用的是 $\max$，与 $a_{t+1}$ 怎么选无关，所以可以用**任意**行为策略产生的数据学习最优策略：**off-policy**。

> **off-policy 对机器人极其重要**：可以复用旧数据、人类示教数据、其它策略的数据。SAC、RLT 都是 off-policy 方法，所以能用 replay buffer，样本效率比 PPO（on-policy）高很多。

### 练习 2：Sarsa 与 Q-learning

两者只有 TD 目标不同。在同一个函数里实现。

In [ ]:
def epsilon_greedy_action(q_row, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(len(q_row)))
    return int(np.argmax(q_row))

def td_control(env, gamma, method="qlearning", alpha=0.1, eps=0.1, n_episodes=500, length=100, seed=0):
    """返回 (q, 每回合总奖励列表)"""
    rng = np.random.default_rng(seed)
    q = np.zeros((env.n_states, env.n_actions))
    ep_rewards = []
    for ep in range(n_episodes):
        s = env.reset(int(rng.integers(env.n_states)))
        a = epsilon_greedy_action(q[s], eps, rng)
        total = 0.0
        for t in range(length):
            ns, r = env.step(a)
            na = epsilon_greedy_action(q[ns], eps, rng)
            # TODO: 根据 method 计算 TD 目标并更新 q[s, a]
            #   sarsa    : target = r + γ q[ns, na]
            #   qlearning: target = r + γ max_a q[ns, a]
            raise NotImplementedError
            total += r
            s, a = ns, na
        ep_rewards.append(total)
    return q, ep_rewards

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def epsilon_greedy_action(q_row, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(len(q_row)))
    return int(np.argmax(q_row))

def td_control(env, gamma, method="qlearning", alpha=0.1, eps=0.1, n_episodes=500, length=100, seed=0):
    """返回 (q, 每回合总奖励列表)"""
    rng = np.random.default_rng(seed)
    q = np.zeros((env.n_states, env.n_actions))
    ep_rewards = []
    for ep in range(n_episodes):
        s = env.reset(int(rng.integers(env.n_states)))
        a = epsilon_greedy_action(q[s], eps, rng)
        total = 0.0
        for t in range(length):
            ns, r = env.step(a)
            na = epsilon_greedy_action(q[ns], eps, rng)
            if method == "sarsa":
                target = r + gamma * q[ns, na]
            else:
                target = r + gamma * q[ns].max()
            q[s, a] -= alpha * (q[s, a] - target)
            total += r
            s, a = ns, na
        ep_rewards.append(total)
    return q, ep_rewards

In [ ]:
# ✅ 自测：先用较大的探索率 ε=0.5
q_sarsa, rew_sarsa = td_control(env, gamma, "sarsa", eps=0.5, n_episodes=1000)
q_ql, rew_ql = td_control(env, gamma, "qlearning", eps=0.5, n_episodes=1000)
err_ql = np.abs(q_ql.max(1) - V_STAR).max()
err_sarsa = np.abs(q_sarsa.max(1) - V_STAR).max()
print(f"Q-learning 的 max_a q 与 v* 的最大误差: {err_ql:.3f}")
print(f"Sarsa      的 max_a q 与 v* 的最大误差: {err_sarsa:.3f}（Sarsa 学的是 ε-greedy 策略自己的值，所以偏低）")
assert err_ql < 0.5
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
env.plot(V=q_sarsa.max(1), policy=q_sarsa.argmax(1), title="Sarsa (eps=0.5): q of the eps-greedy policy", ax=ax[0])
env.plot(V=q_ql.max(1), policy=q_ql.argmax(1), title="Q-learning (eps=0.5): q*", ax=ax[1])
plt.tight_layout(); plt.show()
print("通过 ✅")

**探索不足会怎样？** 把 ε 降到 0.1 再跑一次 Q-learning：

In [ ]:
q_small, _ = td_control(env, gamma, "qlearning", eps=0.1, n_episodes=1000)
print("ε=0.1 时 max_a q 与 v* 的最大误差:", np.abs(q_small.max(1) - V_STAR).max().round(3))
print("右下角 (4,4) 的 q 值:", q_small[24].round(2), "  真实 q*:", Q_STAR[24].round(2))
env.plot(V=q_small.max(1), policy=q_small.argmax(1), title="Q-learning with eps=0.1 (under-explored)"); plt.show()

ε=0.1 时，智能体早早发现了一条"能到达目标的路"，之后绝大多数时间都在走它，更短的路几乎没被尝试过，所以对应的 q 值一直没学好。**Q-learning 理论上的收敛要求每个 (s,a) 被无限次访问**，探索不足时这个前提不成立。真机 RL 里，探索是最贵的东西：每次试错都要花时间、可能损坏设备。这也是 RLT 这类方法"在 VLA 给出的参考动作附近小范围探索"的原因。

### 练习 3：off-policy 的威力：只用随机策略的数据学最优策略

用**完全随机**的行为策略收集一条长轨迹，然后用 Q-learning 从中学习。行为策略从头到尾都是随机的，但学到的贪心策略是最优的。这是 Sarsa 做不到的。

In [ ]:
def qlearning_offline(env, gamma, data, alpha=0.1, n_epochs=20):
    """data: [(s, a, r, s'), ...] 由任意行为策略产生。返回 q"""
    q = np.zeros((env.n_states, env.n_actions))
    # TODO: 多次遍历数据，每条经验做一次 Q-learning 更新
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def qlearning_offline(env, gamma, data, alpha=0.1, n_epochs=20):
    """data: [(s, a, r, s'), ...] 由任意行为策略产生。返回 q"""
    q = np.zeros((env.n_states, env.n_actions))
    for _ in range(n_epochs):
        for s, a, r, ns in data:
            q[s, a] -= alpha * (q[s, a] - (r + gamma * q[ns].max()))
    return q

In [ ]:
# ✅ 自测
rng = np.random.default_rng(0)
s = env.reset(0); data = []
for t in range(20000):                    # 行为策略：完全随机
    a = int(rng.integers(5)); ns, r = env.step(a); data.append((s, a, r, ns)); s = ns
q_off = qlearning_offline(env, gamma, data)
pi_off = policy_to_matrix(q_off.argmax(1))
gap = np.abs(true_v(env, pi_off, gamma) - V_STAR).max()
print(f"仅用随机数据学到的策略，与最优值的最大差距: {gap:.4f}")
assert gap < 1e-3
env.plot(V=q_off.max(1), policy=q_off.argmax(1), title="Q-learning from random data (off-policy)"); plt.show()
print("通过 ✅")

> 这其实就是**离线强化学习**（offline RL）的最简单形式。但有一个前提：数据要**覆盖**所有重要的 $(s,a)$。网格世界很小，随机数据能覆盖；在高维的机器人任务上，示教数据只覆盖很窄的区域。这时 $\max_a q$ 会去"相信"那些从没见过的动作的 q 值（往往被高估），这就是离线 RL 的核心难题：**分布外（OOD）高估**。RLT 对 actor 加 BC 正则、RECAP 用优势条件化而不是 max，都是为了缓解这个问题。

---

## 小结：TD 算法家族

| 算法 | TD 目标 | 求解的方程 | on/off-policy |
|---|---|---|---|
| TD(0) | $r+\gamma v(s')$ | $v_\pi$ 的贝尔曼方程 | on |
| Sarsa | $r+\gamma q(s',a')$ | $q_\pi$ 的贝尔曼方程 | on |
| Expected Sarsa | $r+\gamma\sum_{a'}\pi(a'\mid s')q(s',a')$ | $q_\pi$ 的贝尔曼方程 | on |
| n 步 Sarsa | $r_{t+1}+\dots+\gamma^{n-1}r_{t+n}+\gamma^nq(s_{t+n},a_{t+n})$ | $q_\pi$ 的贝尔曼方程 | on |
| Q-learning | $r+\gamma\max_{a}q(s',a)$ | 贝尔曼**最优**方程 | **off** |

**思考题**：表格方法需要为每个 $(s,a)$ 存一个数。机器人的状态是图像，动作是连续的，有无穷多个，怎么办？答案是**用神经网络近似 q 函数**，见下一课。